In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier
from lightgbm import early_stopping, log_evaluation
from xgboost import XGBClassifier

In [2]:
train_raw = pd.read_csv('home-credit-default-risk/application_train.csv', low_memory=False)
test_raw = pd.read_csv('home-credit-default-risk/application_test.csv', low_memory=False)

# train_imputed = pd.read_csv('imputed_train.csv', low_memory=False)
# test_imputed = pd.read_csv('imputed_test3.csv', low_memory=False)

In [3]:
# Inspect SK_ID_CURR samples
train_ids = train_raw['SK_ID_CURR']
test_ids = test_raw['SK_ID_CURR']
print(train_ids.head())
print(test_ids.head())

0    100002
1    100003
2    100004
3    100006
4    100007
Name: SK_ID_CURR, dtype: int64
0    100001
1    100005
2    100013
3    100028
4    100038
Name: SK_ID_CURR, dtype: int64


#### 找到可以使用的缺失值，不能在data出現過

In [4]:
# import pandas as pd

# # 你可以新增其他想檢查的特殊值
# SPECIAL_VALUES = [-99999]

# def check_special_missing(df: pd.DataFrame, name="df"):
#     print(f"\n===== Checking {name} =====")
    
#     # 找出每個欄位中是否包含特殊值
#     result = {}
#     for col in df.columns:
#         counts = {sv: (df[col] == sv).sum() for sv in SPECIAL_VALUES}
#         # 只報告有出現特殊值的欄位
#         if any(c > 0 for c in counts.values()):
#             result[col] = counts
    
#     if not result:
#         print("✔ No special missing values found.")
#     else:
#         print("⚠ Found special missing values:")
#         for col, counts in result.items():
#             print(f"  {col}: {counts}")

# # 假設你已經有 train_raw, test_raw 的 dataframe
# check_special_missing(train_raw, name="train_raw")
# check_special_missing(test_raw, name="test_raw")

In [5]:
from bagging_tree_ensemble_ft import bagging_tree_ensemble_ft

train_df_bagging, test_df_bagging = bagging_tree_ensemble_ft(
    train_raw, test_raw, 
    label_col='TARGET',              # 你的標籤欄位名
    models=('rf','bagging_dt'),     # 使用 RF 與 Bagging+DT
    n_estimators=80,          # ↓樹數量
    max_leaf_nodes=64,        # ↓每棵樹葉子上限（強烈建議）
    min_samples_leaf=5,       # ↑避免碎葉
    top_ratio=0.05,
    nan_value=-99999,    
) 

c:\Users\487oa\anaconda3\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
c:\Users\487oa\anaconda3\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
c:\Users\487oa\anaconda3\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
c:\Users\487oa\anaconda3\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
c:\Users\487oa\anaconda3\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but DecisionTreeClassifier was fitted without feature names
  warnings.warn(
c:\Users\487oa\anaconda3\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but DecisionTreeClassifier 

In [6]:
import xgboost as xgb
import pandas as pd
from sklearn.model_selection import train_test_split

def run_xgboost_with_valid(train_df, test_df, target_col='TARGET', 
                           id_col='SK_ID_CURR', test_size=0.2, random_state=42):
    """
    使用 XGBoost，包含 validation split、early stopping，最後回傳 test 預測結果。
    train_df: 訓練資料（含 TARGET）
    test_df: 測試資料（不含 TARGET）
    target_col: 目標欄位（預設 TARGET）
    test_size: validation 比例
    """

    # ➤ 分出 X, y
    feature_cols = [col for col in train_df.columns if col not in {target_col, id_col}]
    X = train_df[feature_cols]
    y = train_df[target_col]

    # ➤ 切 validation
    X_train, X_valid, y_train, y_valid = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    # ➤ 做成 DMatrix
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dvalid = xgb.DMatrix(X_valid, label=y_valid)
    dtest  = xgb.DMatrix(test_df[feature_cols])

    # ➤ XGBoost 參數（Home-Credit 常用）
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'eta': 0.05,
        'max_depth': 6,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'seed': random_state
    }

    # ➤ 訓練，含 early stopping
    evals = [(dtrain, 'train'), (dvalid, 'valid')]
    model = xgb.train(
        params,
        dtrain,
        num_boost_round=2000,
        evals=evals,
        early_stopping_rounds=100,
        verbose_eval=100
    )

    print(f"Best iteration: {model.best_iteration}")

    # ➤ 用全部資料重訓
    dfull = xgb.DMatrix(X, label=y)
    final_model = xgb.train(
        params,
        dfull,
        num_boost_round=model.best_iteration
    )

    # ➤ 預測 test
    test_pred = final_model.predict(dtest)

    if id_col in test_df.columns:
        id_values = test_df[id_col].values
    else:
        id_values = np.arange(len(test_df))


    result = pd.DataFrame({
        'SK_ID_CURR': id_values,
        'TARGET': test_pred
    })

    return result


In [11]:
train_df_bagging.to_csv("train_df_bagging.csv") 

In [12]:
test_df_bagging.to_csv("test_df_bagging.csv") 

In [9]:
test_pred = run_xgboost_with_valid(train_df=train_df_bagging, test_df=test_df_bagging) 

[0]	train-auc:0.68317	valid-auc:0.67767
[100]	train-auc:0.74482	valid-auc:0.73452
[200]	train-auc:0.75637	valid-auc:0.73830
[300]	train-auc:0.76596	valid-auc:0.73913
[400]	train-auc:0.77439	valid-auc:0.73906
[425]	train-auc:0.77601	valid-auc:0.73907
Best iteration: 326


In [10]:
test_pred[0].to_csv("answer.csv", index=False) 

KeyError: 0